# 🎴 MTG Tracker — Fine-Tuning Gemma 4 E2B

Execute as células **uma por vez, de cima para baixo**, usando **Shift+Enter**.

**Antes de começar:**
1. Ative a GPU: menu **Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU → Salvar**
2. Faça upload de `dataset_train.jsonl` e `dataset_eval.jsonl` no Google Drive, dentro de uma pasta chamada `MTG_Tracker`

---
## Célula 1 — Verificar GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemError("GPU não detectada! Vá em: Ambiente de execução → Alterar tipo → T4 GPU → Salvar")

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
print(f"✓ GPU: {gpu_name}")
print(f"✓ VRAM: {vram_gb} GB")

if vram_gb < 12:
    print("⚠️  Menos de 12 GB de VRAM — pode dar OOM. Tente reduzir LORA_R para 8 na célula 4.")
else:
    print("✓ VRAM suficiente para Gemma 4 E2B com QLoRA.")

---
## Célula 2 — Conectar ao Google Drive
Uma janela vai pedir permissão — clique em **Permitir**.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/MTG_Tracker"

if not os.path.exists(DRIVE_DIR):
    raise FileNotFoundError(
        f"Pasta '{DRIVE_DIR}' não encontrada no Drive.\n"
        "Crie a pasta 'MTG_Tracker' no Google Drive e faça upload dos arquivos .jsonl."
    )

print(f"✓ Drive conectado: {DRIVE_DIR}")
print("\nArquivos na pasta:")
for f in sorted(os.listdir(DRIVE_DIR)):
    size = os.path.getsize(os.path.join(DRIVE_DIR, f))
    print(f"  • {f}  ({size/1e3:.0f} KB)")

---
## Célula 3 — Instalar Unsloth
Demora ~3 minutos. O `%%capture` esconde o output — é normal não aparecer nada.

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade trl datasets transformers accelerate

In [ ]:
import unsloth
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
print(f"✓ Unsloth {unsloth.__version__} pronto.")

---
## Célula 4 — Carregar modelo Gemma 4 E2B
Baixa ~2–3 GB. Demora 3–5 minutos.

In [ ]:
import torch
from unsloth import FastLanguageModel

MODEL_ID    = "unsloth/gemma-4-E2B-it"
MAX_SEQ_LEN = 512
LORA_R      = 16   # reduza para 8 se der erro de memória (OOM)

print(f"▶ Carregando {MODEL_ID} ...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_ID,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_R,
    lora_alpha                 = LORA_R * 2,
    lora_dropout               = 0,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                   "gate_proj", "up_proj", "down_proj"],
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
vram_used = torch.cuda.memory_allocated() / 1e9

print(f"✓ Modelo carregado.")
print(f"  Parâmetros treináveis: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
print(f"  VRAM usada: {vram_used:.1f} GB")

---
## Célula 5 — Carregar e formatar dataset

In [ ]:
import os
from datasets import load_dataset

DRIVE_DIR     = "/content/drive/MyDrive/MTG_Tracker"
DATASET_TRAIN = f"{DRIVE_DIR}/dataset_train.jsonl"
DATASET_EVAL  = f"{DRIVE_DIR}/dataset_eval.jsonl"

for path in [DATASET_TRAIN, DATASET_EVAL]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Arquivo não encontrado: {path}\n"
            "Faça upload dele na pasta MTG_Tracker do Google Drive."
        )

train_ds = load_dataset("json", data_files=DATASET_TRAIN, split="train")
eval_ds  = load_dataset("json", data_files=DATASET_EVAL,  split="train")

def format_example(example):
    return {"text": tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )}

train_ds = train_ds.map(format_example, remove_columns=train_ds.column_names)
eval_ds  = eval_ds.map(format_example,  remove_columns=eval_ds.column_names)

print(f"✓ Dataset pronto.")
print(f"  Treino: {len(train_ds):,} exemplos")
print(f"  Eval:   {len(eval_ds):,} exemplos")
print(f"\nExemplo (300 chars):")
print(train_ds[0]["text"][:300])

---
## Célula 6 — Treinar
**Demora ~1–2h.** Os checkpoints são salvos diretamente no Google Drive a cada 100 steps — se a sessão cair, nada se perde.

Acompanhe o `eval/loss` — esperamos que caia para ~0.05–0.15.

In [ ]:
import torch, os
from trl import SFTTrainer, SFTConfig

DRIVE_DIR  = "/content/drive/MyDrive/MTG_Tracker"
OUTPUT_DIR = f"{DRIVE_DIR}/checkpoints"   # ← salva direto no Drive
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE = 2
GRAD_ACCUM = 8

sft_config = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = 3,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = 2e-4,
    warmup_steps                = 50,
    lr_scheduler_type           = "cosine",
    optim                       = "adamw_8bit",
    bf16                        = torch.cuda.is_bf16_supported(),
    fp16                        = not torch.cuda.is_bf16_supported(),
    logging_steps               = 10,
    save_steps                  = 100,
    eval_steps                  = 100,
    eval_strategy               = "steps",
    save_total_limit            = 3,
    load_best_model_at_end      = True,
    report_to                   = "none",
    dataloader_num_workers      = 2,
    remove_unused_columns       = False,
    dataset_text_field          = "text",
)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_ds,
    eval_dataset  = eval_ds,
    args          = sft_config,
)

steps_total = len(train_ds) * 3 // (BATCH_SIZE * GRAD_ACCUM)
print(f"▶ Iniciando treino...")
print(f"  Steps totais: ~{steps_total}")
print(f"  Checkpoints no Drive: {OUTPUT_DIR}")
print()

stats = trainer.train()

print()
print("✓ Treino concluído!")
print(f"  Tempo:      {stats.metrics['train_runtime']/60:.1f} minutos")
print(f"  Loss final: {stats.metrics['train_loss']:.4f}")

---
## Célula 7 — Salvar adapter final no Drive

In [ ]:
import os

DRIVE_DIR    = "/content/drive/MyDrive/MTG_Tracker"
ADAPTER_PATH = f"{DRIVE_DIR}/adapter_final"
os.makedirs(ADAPTER_PATH, exist_ok=True)

print("▶ Salvando adapter final...")
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

files = os.listdir(ADAPTER_PATH)
print(f"✓ Adapter salvo em: {ADAPTER_PATH}")
print(f"  Arquivos ({len(files)}): {', '.join(files)}")

---
## Célula 8 — Exportar modelo final em GGUF

Esta célula:
1. Instala o llama.cpp (se ainda não estiver instalado)
2. Mescla o adapter com o modelo base e salva em HuggingFace format no Drive
3. Converte para f16 GGUF **localmente** (SSD rápido do Colab, ~9 GB)
4. Quantiza para Q4_K_M localmente (~1.5 GB)
5. Copia **só** o Q4_K_M final para o Drive

**Verifica erros reais** — não vai imprimir ✓ se algo falhar.

Demora ~20–30 minutos no total.

In [ ]:
import os
import subprocess
import shutil

# ── Caminhos ──────────────────────────────────────────────────────────────────
DRIVE_DIR        = "/content/drive/MyDrive/MTG_Tracker"
MERGED_DIR       = f"{DRIVE_DIR}/merged_model"   # HF format (salvo no Drive)
LLAMA_DIR        = "/root/llama.cpp"
GGUF_F16_LOCAL   = "/content/mtg_f16.gguf"       # intermediário LOCAL (rápido)
GGUF_Q4KM_LOCAL  = "/content/mtg_q4km.gguf"      # quantizado LOCAL
GGUF_Q4KM_DRIVE  = f"{DRIVE_DIR}/mtg_extractor_q4km.gguf"  # destino final

F16_MIN_SIZE_GB = 8.5   # se o f16 tiver menos que isso, está incompleto


def run(cmd, desc):
    """Executa comando shell e lança exceção se falhar."""
    print(f"  ▶ {desc}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  STDOUT: {result.stdout[-2000:]}")
        print(f"  STDERR: {result.stderr[-2000:]}")
        raise RuntimeError(f"FALHOU ({result.returncode}): {desc}")
    return result.stdout


def file_size_gb(path):
    return os.path.getsize(path) / 1e9 if os.path.exists(path) else 0


# ── Passo 1: Instalar llama.cpp ───────────────────────────────────────────────
print("[1/5] Verificando llama.cpp...")
if not os.path.isdir(LLAMA_DIR):
    run(f"git clone https://github.com/ggerganov/llama.cpp {LLAMA_DIR}",
        "Clonando llama.cpp")
    run(f"pip install -q -r {LLAMA_DIR}/requirements.txt",
        "Instalando dependências")
else:
    print("  ✓ llama.cpp já instalado.")

# Garantir que o binário llama-quantize existe
quantize_bin = f"{LLAMA_DIR}/llama-quantize"
if not os.path.exists(quantize_bin):
    print("  Compilando llama-quantize...")
    run(f"cd {LLAMA_DIR} && cmake -B build -DLLAMA_CURL=OFF -DGGML_CUDA=OFF > /dev/null 2>&1 && "
        f"cmake --build build --config Release -j$(nproc) --target llama-quantize 2>&1 | tail -5",
        "Compilando")
    run(f"cp {LLAMA_DIR}/build/bin/llama-quantize {quantize_bin}", "Copiando binário")
print("  ✓ llama-quantize pronto.")


# ── Passo 2: Mesclar adapter + modelo base ────────────────────────────────────
print("\n[2/5] Mesclando adapter com modelo base...")
if os.path.isdir(MERGED_DIR) and len(os.listdir(MERGED_DIR)) > 3:
    print(f"  ✓ Modelo mesclado já existe em {MERGED_DIR}, pulando.")
else:
    os.makedirs(MERGED_DIR, exist_ok=True)
    from unsloth import FastLanguageModel

    ADAPTER_PATH = f"{DRIVE_DIR}/adapter_final"
    if not os.path.isdir(ADAPTER_PATH):
        raise FileNotFoundError(
            f"Adapter não encontrado em {ADAPTER_PATH}.\n"
            "Execute a célula 7 primeiro."
        )

    print("  Carregando adapter (pode demorar 3–5 min)...")
    _model, _tokenizer = FastLanguageModel.from_pretrained(
        model_name     = ADAPTER_PATH,
        max_seq_length = 512,
        dtype          = None,
        load_in_4bit   = True,
    )
    print("  Salvando modelo mesclado no Drive (pode demorar 5–10 min)...")
    _model.save_pretrained_merged(MERGED_DIR, _tokenizer, save_method="merged_16bit")
    print(f"  ✓ Mesclado em: {MERGED_DIR}")


# ── Passo 3: Converter para f16 GGUF (LOCAL) ──────────────────────────────────
print("\n[3/5] Convertendo para f16 GGUF (local)...")
f16_size = file_size_gb(GGUF_F16_LOCAL)
if f16_size >= F16_MIN_SIZE_GB:
    print(f"  ✓ f16 já existe e parece completo ({f16_size:.1f} GB), pulando conversão.")
else:
    if f16_size > 0:
        print(f"  ⚠️  f16 incompleto ({f16_size:.1f} GB < {F16_MIN_SIZE_GB} GB esperado). Removendo e reconvertendo...")
        os.remove(GGUF_F16_LOCAL)

    convert_script = f"{LLAMA_DIR}/convert_hf_to_gguf.py"
    run(
        f"python {convert_script} {MERGED_DIR} "
        f"--outfile {GGUF_F16_LOCAL} --outtype f16",
        "Convertendo HF → f16 GGUF"
    )

    f16_size = file_size_gb(GGUF_F16_LOCAL)
    if f16_size < F16_MIN_SIZE_GB:
        raise RuntimeError(
            f"f16 gerado muito pequeno ({f16_size:.1f} GB). "
            "Conversão falhou silenciosamente."
        )
    print(f"  ✓ f16 criado: {f16_size:.1f} GB")


# ── Passo 4: Quantizar para Q4_K_M (LOCAL) ────────────────────────────────────
print("\n[4/5] Quantizando para Q4_K_M (local)...")
if os.path.exists(GGUF_Q4KM_LOCAL) and file_size_gb(GGUF_Q4KM_LOCAL) > 1.0:
    q4_size = file_size_gb(GGUF_Q4KM_LOCAL)
    print(f"  ✓ Q4_K_M já existe ({q4_size:.1f} GB), pulando quantização.")
else:
    if os.path.exists(GGUF_Q4KM_LOCAL):
        os.remove(GGUF_Q4KM_LOCAL)

    run(
        f"{quantize_bin} {GGUF_F16_LOCAL} {GGUF_Q4KM_LOCAL} Q4_K_M",
        "Quantizando f16 → Q4_K_M"
    )

    q4_size = file_size_gb(GGUF_Q4KM_LOCAL)
    if q4_size < 1.0:
        raise RuntimeError(
            f"Q4_K_M gerado muito pequeno ({q4_size:.1f} GB). "
            "Quantização falhou."
        )
    print(f"  ✓ Q4_K_M criado: {q4_size:.1f} GB")

    # Libera espaço local removendo o f16 (só depois que Q4_K_M está confirmado)
    os.remove(GGUF_F16_LOCAL)
    print(f"  ✓ f16 local removido (libera espaço).")


# ── Passo 5: Copiar Q4_K_M para o Drive ───────────────────────────────────────
print("\n[5/5] Copiando Q4_K_M para o Drive (~1.5 GB)...")
if os.path.exists(GGUF_Q4KM_DRIVE):
    existing_size = file_size_gb(GGUF_Q4KM_DRIVE)
    local_size    = file_size_gb(GGUF_Q4KM_LOCAL)
    if abs(existing_size - local_size) < 0.05:  # diferença < 50 MB = mesmo arquivo
        print(f"  ✓ Já existe no Drive ({existing_size:.1f} GB), pulando cópia.")
    else:
        print(f"  ⚠️  Arquivo no Drive parece diferente ({existing_size:.1f} GB vs {local_size:.1f} GB). Substituindo...")
        shutil.copy2(GGUF_Q4KM_LOCAL, GGUF_Q4KM_DRIVE)
else:
    shutil.copy2(GGUF_Q4KM_LOCAL, GGUF_Q4KM_DRIVE)

final_size = file_size_gb(GGUF_Q4KM_DRIVE)
print(f"\n{'='*60}")
print(f"✅ CONCLUÍDO!")
print(f"   Arquivo: {GGUF_Q4KM_DRIVE}")
print(f"   Tamanho: {final_size:.2f} GB")
print(f"   Próximo passo: baixe o arquivo do Drive e integre no app.")
print(f"{'='*60}")

---
## Célula 9 — Testar o modelo
Testa com 3 exemplos reais para confirmar que está funcionando.

In [ ]:
import torch
from unsloth import FastLanguageModel

# Recarrega o modelo do adapter salvo no Drive (funciona mesmo após restart de sessão)
DRIVE_DIR    = "/content/drive/MyDrive/MTG_Tracker"
ADAPTER_PATH = f"{DRIVE_DIR}/adapter_final"
MAX_SEQ_LEN  = 512

print(f"▶ Carregando modelo de {ADAPTER_PATH} ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = ADAPTER_PATH,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)

SYSTEM_PROMPT = (
    "You are a Magic: The Gathering match data extractor. "
    "Given a natural language match description (in any language), "
    "extract structured data as JSON.\n\n"
    "Fields: won (bool), drew (bool), myDeck (str|null), oppDeck (str|null), "
    "format (Commander|Modern|Standard|Pioneer|Legacy|Pauper|Other|null), "
    "onPlay (bool|null), archetype (Aggro|Midrange|Control|Combo|Stax|null).\n"
    "Respond with ONLY valid JSON. No explanation, no markdown."
)

FastLanguageModel.for_inference(model)

testes = [
    "Venci o Tron com meu Rhinos no Modern, fui primeiro.",
    "Lost to Hammer Time in Modern, was on the draw.",
    "Ganhei de Najeela com Blue Farm no cEDH.",
]

print("🧪 Testando modelo:\n" + "=" * 60)

for texto in testes:
    inputs = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user",   "content": texto}],
        return_tensors="pt",
        add_generation_prompt=True,
    ).to("cuda")

    with torch.no_grad():
        out = model.generate(inputs, max_new_tokens=150, temperature=0.1, do_sample=True)

    resposta = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"Input:  {texto}")
    print(f"Output: {resposta.strip()}")
    print("-" * 60)

print("\n✓ Se os JSONs estiverem corretos, o modelo está pronto!")

---
## ✅ Concluído!

O arquivo `mtg_extractor_q4km.gguf` está na pasta `MTG_Tracker` do seu Google Drive.

**Próximos passos no projeto React Native:**
```bash
npx expo install llama.rn
```
Depois substituir o mock em `HomeScreen.tsx` pela inferência real com `llama.rn`.